# Heston GPU Monte Carlo — Colab quick run

**Runtime → Change runtime type → GPU** (T4 / L4 / …).

If this notebook lives in a **subfolder** of a monorepo, set `PROJECT_SUBDIR` (see cell 2).

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from pathlib import Path

# --- edit for your GitHub repository ---
REPO_URL = "https://github.com/chezke/Monte-Carlo-simulation-of-Heston-model.git"
BRANCH = "main"
# If repo root is *not* this project, cd into it after clone (use "" if notebook lives at repo root):
PROJECT_SUBDIR = ""  # example: "Monte Carlo simulation of Heston model" or "heston-mc"

CLONE_DIR = "/content/heston_mc_src"
SAFE_DIR = "/content"

def safe_chdir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    print("cwd:", os.getcwd())
safe_chdir(SAFE_DIR)

if os.path.isdir(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, CLONE_DIR],
    check=True,
)
os.chdir(CLONE_DIR)
if PROJECT_SUBDIR:
    os.chdir(PROJECT_SUBDIR)
print("cwd:", os.getcwd())
!ls -la

In [ ]:
# Colab GPU images usually ship nvcc; if 'make' fails, try: which nvcc
!make clean 2>/dev/null; make NVFLAGS="-O3 -std=c++14 -Iinclude"

In [ ]:
!./bin/MC_Euler

In [ ]:
!./bin/MC_exact

In [ ]:
!./bin/MC_benchmark_Q3 > results_q3.csv

In [ ]:
# Q3 sweep can be slow; reduce grid/path count via NVFLAGS if needed
# !make NVFLAGS="-O3 -std=c++14 -Iinclude -DHESTON_Q3_N_PATHS=16384" bin/MC_benchmark_Q3
!./bin/MC_benchmark_Q3 | head -20

## Q3 Analysis — Euler vs Almost-exact

We focus on three points:
1. Runtime
2. Error vs Broadie–Kaya reference (`err_* = mean_* - mean_exact`)
3. Effect of coarse step size (`Δt = 1/30`) in the almost-exact scheme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results_q3.csv", comment="#")
print(f"Loaded {len(df)} parameter combinations (Feller-filtered)")
df.head()

### 1. Runtime comparison

Euler uses simple per-step arithmetic.
BK-exact and almost-exact require Poisson + Gamma sampling each step, so at the same `Δt` they should be much slower.

In [ ]:
from q3_plots import plot_execution_time, plot_error_analysis, plot_dt_impact
plot_execution_time(df)

### 2. Error comparison

`err_* = mean_* - mean_exact`, with BK (`Δt = 1/1000`) as the MC reference.
So each `err_*` contains both discretization error and residual MC noise.

In [ ]:
plot_error_analysis(df)

### 3. Effect of `Δt = 1/30` in almost-exact

Using fewer steps gives a large runtime gain.
The tradeoff is larger log-price discretization error, so `|err|` usually increases.

Below we compare `Δt = 1/1000` and `Δt = 1/30` directly.

In [ ]:
plot_dt_impact(df)

### 4. Key takeaways

- **Speed:** Euler is fastest.
- **Costly methods:** BK and almost-exact (`Δt = 1/1000`) have similar runtime and are much slower than Euler.
- **Coarser almost-exact:** `Δt = 1/30` is much faster than `Δt = 1/1000`, but usually still slower than Euler.
- **Accuracy:** almost-exact (`Δt = 1/1000`) has the smallest `|err|` on average; Euler is larger; almost-exact (`Δt = 1/30`) is largest.
- **Practical choice:** use `Δt = 1/30` for speed-focused runs; use fine `Δt` (or BK) when accuracy is the priority.